In [1]:
import torch
import numpy as np
import mdtraj as md
import h5py

from openmm.app import *
from openmm import *
import openmm.unit as unit
from pathlib import Path
from sys import stdout
from boltz.model.modules.diffusion import DiffusionModule
from boltz.model.models.boltz1 import Boltz1
from dataclasses import asdict, dataclass

## Loading Boltz for force calculation.

In [2]:
recycling_steps = 3
diffusion_sampling_steps = 200
diffusion_samples = 1
max_parallel_samples = 5
confidence = False       
write_full_pae = True
write_full_pde = False

predict_args = {
    "recycling_steps": recycling_steps,
    "sampling_steps": diffusion_sampling_steps,
    "diffusion_samples": diffusion_samples,
    "max_parallel_samples": max_parallel_samples,
    "write_confidence_summary": confidence,
    "write_full_pae": write_full_pae,
    "write_full_pde": write_full_pde,
}

@dataclass
class BoltzDiffusionParams:
    """Diffusion process parameters."""

    gamma_0: float = 0.605
    gamma_min: float = 1.107
    noise_scale: float = 0.901
    rho: float = 8
    step_scale: float = 1.638
    sigma_min: float = 0.0004
    sigma_max: float = 160.0
    sigma_data: float = 16.0
    P_mean: float = -1.2
    P_std: float = 1.5
    coordinate_augmentation: bool = True
    alignment_reverse_diff: bool = True
    synchronize_sigmas: bool = True
    use_inference_model_cache: bool = True

@dataclass
class PairformerArgs:
    """Pairformer arguments."""

    num_blocks: int = 48
    num_heads: int = 16
    dropout: float = 0.0
    activation_checkpointing: bool = False
    offload_to_cpu: bool = False
    v2: bool = False

diffusion_params = BoltzDiffusionParams()
step_scale = 1.638
diffusion_params.step_scale = step_scale
pairformer_args = PairformerArgs()

no_kernels = False

@dataclass
class MSAModuleArgs:
    """MSA module arguments."""

    msa_s: int = 64
    msa_blocks: int = 4
    msa_dropout: float = 0.0
    z_dropout: float = 0.0
    use_paired_feature: bool = True
    pairwise_head_width: int = 32
    pairwise_num_heads: int = 4
    activation_checkpointing: bool = False
    offload_to_cpu: bool = False
    subsample_msa: bool = False
    num_subsampled_msa: int = 1024

subsample_msa = True
num_subsampled_msa = 1024

msa_args = MSAModuleArgs(
    subsample_msa=subsample_msa,
    num_subsampled_msa=num_subsampled_msa,
    use_paired_feature=False,
)

@dataclass
class BoltzSteeringParams:
    """Steering parameters."""

    fk_steering: bool = True
    num_particles: int = 3
    fk_lambda: float = 4.0
    fk_resampling_interval: int = 3
    guidance_update: bool = True
    num_gd_steps: int = 20

use_potentials = False

steering_args = BoltzSteeringParams()
steering_args.fk_steering = use_potentials
steering_args.guidance_update = use_potentials

In [3]:
model_cls = Boltz1
cache = Path("~/.boltz/").expanduser()
checkpoint = cache / "boltz1_noconfidence.ckpt"

boltz = model_cls.load_from_checkpoint(
    checkpoint,
    strict=False,
    predict_args=predict_args,
    map_location="cpu",
    diffusion_process_args=asdict(diffusion_params),
    ema=False,
    use_kernels=not no_kernels,
    pairformer_args=asdict(pairformer_args),
    msa_args=asdict(msa_args),
    steering_args=asdict(steering_args),
    confidence_prediction=False
)
boltz.eval()

Boltz1(
  (lddt): ModuleDict(
    (dna_protein): MeanMetric()
    (rna_protein): MeanMetric()
    (ligand_protein): MeanMetric()
    (dna_ligand): MeanMetric()
    (rna_ligand): MeanMetric()
    (intra_ligand): MeanMetric()
    (intra_dna): MeanMetric()
    (intra_rna): MeanMetric()
    (intra_protein): MeanMetric()
    (protein_protein): MeanMetric()
    (modified): MeanMetric()
    (pocket_ligand_protein): MeanMetric()
  )
  (disto_lddt): ModuleDict(
    (dna_protein): MeanMetric()
    (rna_protein): MeanMetric()
    (ligand_protein): MeanMetric()
    (dna_ligand): MeanMetric()
    (rna_ligand): MeanMetric()
    (intra_ligand): MeanMetric()
    (intra_dna): MeanMetric()
    (intra_rna): MeanMetric()
    (intra_protein): MeanMetric()
    (protein_protein): MeanMetric()
    (modified): MeanMetric()
    (pocket_ligand_protein): MeanMetric()
  )
  (complex_lddt): ModuleDict(
    (dna_protein): MeanMetric()
    (rna_protein): MeanMetric()
    (ligand_protein): MeanMetric()
    (dna_ligand

## Loading Boltz args.

In [15]:
with h5py.File('/home/ethanz/boltz-likelihoods/conditioning/chignolin/tensors.hdf5') as f:
    s = torch.from_numpy(f['s_trunk'][:]).to(boltz.device)
    z = torch.from_numpy(f['z_trunk'][:]).to(boltz.device)
    s_inputs = torch.from_numpy(f['s_inputs'][:]).to(boltz.device)
    relative_position_encoding = torch.from_numpy(f['rpes'][:]).to(boltz.device)
    pdistogram = torch.from_numpy(f['pdistogram'][:]).to(boltz.device)

cond_args = {
    "s_trunk": s,
    "z_trunk": z,
    "s_inputs": s_inputs,
    "relative_position_encoding": relative_position_encoding,
}

sigmas = boltz.structure_module.sample_schedule(num_sampling_steps=200)
gammas = torch.where(sigmas > boltz.structure_module.gamma_min, boltz.structure_module.gamma_0, 0.0)
sigmas_and_gammas = list(zip(sigmas[:-1], sigmas[1:], gammas[1:]))

sigma_tm, _, gamma = sigmas_and_gammas[170] # TODO: hardcoded.
sigma_tm, gamma = sigma_tm.item(), gamma.item()
t_hat = sigma_tm * (1 + gamma) # Constant noise level for score calcs.
print(t_hat, type(t_hat))

0.25502604246139526 <class 'float'>


## Preparing OpenMM system.

In [3]:
# Testing raw initialization using coords, types, and massses.
traj = md.load_pdb("/home/ethanz/boltz-likelihoods/conditioning/chignolin/chignolin_folded.pdb")

# 1. Coordinates (nm by default in mdtraj, shape: (n_frames, n_atoms, 3))
coords = traj.xyz[0]  # first frame only, shape (n_atoms, 3)

# 2. Elements (mdtraj stores Atom objects with element info)
elements = [atom.element.symbol for atom in traj.topology.atoms]

# 3. Masses (if you want them)
masses = [atom.element.mass for atom in traj.topology.atoms]

system = System()
particles = []
for m in masses:
    particles.append(system.addParticle(m * unit.dalton))  # add particles with given mass

# 2. Make a barebones Topology (optional but useful if you want to save trajectories)
topology = Topology()
chain = topology.addChain()
residue = topology.addResidue("RES", chain)
atoms = []
for i, m in enumerate(masses):
    atom = topology.addAtom(f"A{i}", element=None, residue=residue)  # element can be None
    atoms.append(atom)

In [6]:
pdb = PDBFile("/home/ethanz/boltz-likelihoods/conditioning/chignolin/chignolin_folded.pdb")
forcefield = ForceField("amber14-all.xml")  # Arbitrary ff. 
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=NoCutoff,
    constraints=None
)

# Removing all forces - we only want to use our score forces.
while system.getNumForces() > 0:
    system.removeForce(0)

n_atoms = system.getNumParticles()

# Setting up umbrella force. 
umbrella_force = CustomExternalForce("fx*x + fy*y + fz*z")
umbrella_force.addPerParticleParameter("fx")
umbrella_force.addPerParticleParameter("fy")
umbrella_force.addPerParticleParameter("fz")
for i in range(n_atoms):
    umbrella_force.addParticle(i, [0.0, 0.0, 0.0])
system.addForce(umbrella_force)

def calc_umbrella_energy(coords_nm, cv0, k_umb, calc_cv):
    """Calculate the umbrella energy based on current coordinates.

    Parameters
    ----------
    coords_nm : torch.tensor
        Coordinates in nanometers, shape (n_atoms, 3). Needs to be 
        differentiable.
    cv0 : torch.tensor
        Center of the umbrella potential in CV space, shape (n_cvs,).
    k_umb : float
        Spring constant for the umbrella potential.
    calc_cv : ChignolinCV functor object
        Call to calculate CVs from coordinates.
    """
    cv = calc_cv(coords_nm)
    diff = cv - cv0
    return 0.5 * k_umb * torch.sum(diff ** 2)


# Setting up score module force.
nn_force = CustomExternalForce("-fx*x - fy*y - fz*z")
nn_force.addPerParticleParameter("fx")
nn_force.addPerParticleParameter("fy")
nn_force.addPerParticleParameter("fz")

for i in range(n_atoms):
    nn_force.addParticle(i, [0.0, 0.0, 0.0])  # init with 0 force
system.addForce(nn_force)


# Setting up integrator and context.
dt = 0.002 * unit.picoseconds
temperature = 300 * unit.kelvin
friction = 1.0 / unit.picosecond

integrator = LangevinIntegrator(temperature, friction, dt)
platform = Platform.getPlatformByName("CPU")
context = Context(system, integrator, platform)
init_positions = np.zeros((n_atoms, 3))
context.setPositions(init_positions)

## CV for a particular system.

In [7]:
EIGENS_PATH = "/home/ethanz/data_boltz_likelihood/ticas/chignolin_tica_eigenvectors.npy"
MEANS_PATH = "/home/ethanz/data_boltz_likelihood/ticas/chignolin_tica_mean.npy"

tica_eigenvectors = np.load(EIGENS_PATH)
tica_means = np.load(MEANS_PATH)

class ChignolinCV:
    """Functor to compute the first two tICA coordinates for Chignolin 
    based on CA atom distances.

    Parameters
    ----------
    top : md.Topology
        mdtraj topology of the PDB.
    """
    def __init__(self, top, mean, eigenvectors, device="cpu"):
        self.device = device
        self.mean = torch.tensor(mean, dtype=torch.float32, device=device)
        self.eigenvectors = torch.tensor(eigenvectors, dtype=torch.float32, device=device)

        # Select CA atoms and precompute distance indices
        self.ca_indices = top.select("name CA")
        traj = md.Trajectory(np.zeros((1, top.n_atoms, 3)), top)
        traj = traj.atom_slice(self.ca_indices)
        self.distance_indices = np.array(
            [
                [i, j] for i in range(traj.n_atoms)
                for j in range(i + 1, traj.n_atoms)
            ],
            dtype=np.int32
        )

    def __call__(self, coords_nm):
        """
        Compute the tICA coordinates based on CA atom distances.

        Parameters
        ----------
        coords_nm: torch.Tensor of shape (n_atoms, 3) 
            System coordinates in nm.

        Returns
        -------
        torch.Tensor of shape (2,) 
            The first two tICA coordinates.
        """
        # Select CA coords
        ca_coords = coords_nm[self.ca_indices]  # (n_CA, 3)

        # Compute pairwise distances
        # Broadcasting to get (n_pairs,)
        diffs = ca_coords[self.distance_indices[:, 0]] - ca_coords[self.distance_indices[:, 1]]
        distances = torch.linalg.norm(diffs, dim=1)

        # tICA projection
        proj = (distances - self.mean) @ self.eigenvectors
        return proj[:2]  # first 2 tICs

In [8]:
# Sanity checking CV implementation
def get_chignolin_reaction_coordinates(
    top, 
    r_coordinates,
    mean, 
    eigenvectors,
    device="cuda:0"
):
    """Computes Chignolin RC values for each frame of a trajectory.

    Parameters
    ----------
    top : mdtraj.Topology object
        Topology of the trajectory.

    r_coordinates : np.array of shape (frames, n_atoms, 3)
        Coordinate array of the trajectory.

    Returns
    -------
    all_projections_sub : torch.tensor of shape (frames, 2)
        tICA coordinate values for every frame.
    """
    traj = md.Trajectory(r_coordinates, top).atom_slice(top.select("name CA"))
    distance_indeces = [[i, j] for i in range(traj.n_atoms) 
                    for j in range(i + 1, traj.n_atoms)]
    all_distances = md.compute_distances(traj, distance_indeces)
    all_projections_sub = ((all_distances - mean)  @ eigenvectors)[:, :2]
    return torch.tensor(all_projections_sub)

# Checking vs. Abby's implementation
md_pdb_folded = md.load("/home/ethanz/boltz-likelihoods/conditioning/chignolin/chignolin_folded.pdb")
md_pdb_unfolded = md.load("/home/ethanz/boltz-likelihoods/conditioning/chignolin/chignolin_unfolded.pdb")

for pdb in [md_pdb_folded, md_pdb_unfolded]:
    init_coords_nm = torch.from_numpy(pdb.xyz[0])  # Get initial coordinates in nm
    calc_cv = ChignolinCV(top=pdb.topology, mean=tica_means, eigenvectors=tica_eigenvectors, device="cpu")
    
    cv0 = calc_cv(init_coords_nm)
    cv_gt = get_chignolin_reaction_coordinates(
        top=pdb.topology,
        r_coordinates=pdb.xyz,
        mean=tica_means,
        eigenvectors=tica_eigenvectors
    )
    print(cv0)
    print(cv_gt)

tensor([ 1.5998, -0.8001])
tensor([[ 1.5998, -0.8001]], dtype=torch.float64)
tensor([-0.4991,  0.0011])
tensor([[-0.4991,  0.0011]], dtype=torch.float64)


In [9]:
def dummy_force_model(coords):
    return coords

In [ ]:
n_steps = 100
k_umb = 10

for step in range(n_steps):
    # Get positions from OpenMM.
    state = context.getState(getPositions=True)
    coords_nm = torch.tensor(
        state.getPositions(asNumpy=True).value_in_unit(unit.nanometer),
        dtype=torch.float32
    )

    # Computing score module 'force' and adding them to system. 
    f_nn = dummy_force_model(coords_nm)

    for i in range(n_atoms):
        nn_force.setParticleParameters(i, i, f_nn[i].tolist())
    nn_force.updateParametersInContext(context)

    '''
    To compute umbrella forces, we need to use chain rule to relate 
    changes in CV to changes in atomic positions.

    Since we use a harmonic potential for the umbrella bias in CV space,
    we have dU/dCV = k_umb * (CV - CV0), where CV0 is the window center.

    Then, we need dCV/dR, which we compute using torch grad.
    '''
    
    cv = calc_cv(coords_nm)
    diff = cv - cv0
    dU_dCV = k_umb * diff  # shape (n_cvs,)
    
    f_umb = torch.zeros_like(coords_nm)
    for atom in range(n_atoms):
        for dim in range(3):
            coords_p = coords_nm.clone()
            coords_m = coords_nm.clone()
            coords_p[atom, dim] += eps
            coords_m[atom, dim] -= eps
            cv_p = calc_cv(coords_p)
            cv_m = calc_cv(coords_m)
            dCV_dR = (cv_p - cv_m) / (2 * eps)  # shape (n_cvs,)
            # Force contribution from umbrella bias = -Σ_j dU/dCV_j * dCV_j/dR
            f_umb[atom, dim] = -torch.sum(dU_dCV * dCV_dR)

    for i in range(n_atoms):
        umbrella_force.setParticleParameters(i, i, f_umb[i].tolist())
    umbrella_force.updateParametersInContext(context)

    # ---- Integrate ----
    integrator.step(1)

    if step % 10 == 0:
        print(f"Step {step} | CV = {cv.numpy()} | diff^2 sum = {torch.sum(diff**2).item():.4f}")

TypeError: DiffusionModule.forward() missing 1 required positional argument: 'feats'